# Task 02 — EDA and Insight Generation
**Dataset:** NYC Airbnb 2019  
**Input:** `../task_01/outputs/ingested.csv`  
**Goal:** Understand the target variable (`price`), explore features, identify relationships, and produce an analysis-ready dataset.  
**SEED:** 42

In [ ]:
# ── Imports & Setup ──────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

SEED = 42
np.random.seed(SEED)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 13})

# ── Robust project-root finder (works regardless of kernel CWD) ──────────────
def _find_project_root() -> Path:
    for candidate in [Path.cwd()] + list(Path.cwd().parents):
        if (candidate / 'data' / 'raw').exists():
            return candidate
    raise FileNotFoundError('Cannot find project root — expected a parent containing data/raw/')

PROJECT_ROOT = _find_project_root()
INPUT_PATH   = PROJECT_ROOT / 'agents' / 'claude' / 'task_01' / 'outputs' / 'ingested.csv'
OUTPUT_DIR   = PROJECT_ROOT / 'agents' / 'claude' / 'task_02' / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Input        : {INPUT_PATH}')
print(f'Output dir   : {OUTPUT_DIR}')

In [ ]:
# ── Load Data ────────────────────────────────────────────────────────────────
df = pd.read_csv(INPUT_PATH)
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
df.info()
df.describe()

---
## 1. Target Variable — `price`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw price
axes[0].hist(df['price'], bins=100, color='steelblue', edgecolor='white')
axes[0].set_title('Price Distribution (raw)')
axes[0].set_xlabel('Price (USD per night)')
axes[0].set_ylabel('Count')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${int(x)}'))

# Log-transformed price
axes[1].hist(np.log1p(df['price']), bins=80, color='darkorange', edgecolor='white')
axes[1].set_title('Price Distribution (log1p-transformed)')
axes[1].set_xlabel('log1p(Price)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'price_distribution.png')
plt.show()
print(f'Saved → price_distribution.png')

In [ ]:
print('Price summary stats:')
print(df['price'].describe().round(2))
print(f'\nSkewness (raw)     : {df["price"].skew():.3f}')
print(f'Skewness (log1p)   : {np.log1p(df["price"]).skew():.3f}')
print(f'Listings > $500    : {(df["price"] > 500).sum()} ({(df["price"] > 500).mean()*100:.1f}%)')
print(f'Listings > $1000   : {(df["price"] > 1000).sum()} ({(df["price"] > 1000).mean()*100:.1f}%)')

### Interpretation — Price Distribution

The raw `price` distribution is **strongly right-skewed** with a long tail of expensive listings. The bulk of listings sit below \$300/night, but extreme values (>\$1,000) pull the mean well above the median.

**Key finding:** The log1p-transformed distribution is approximately normal and centred around log1p(price) ≈ 5 (~\$150). This strongly suggests that **models in Task 03 should train on `log1p(price)` as the target** and back-transform predictions with `expm1()` for evaluation. Training on raw price would penalise outliers disproportionately and hurt generalisation.

In [ ]:
# Box plot by room type
fig, ax = plt.subplots(figsize=(10, 5))
order = df.groupby('room_type')['price'].median().sort_values(ascending=False).index
sns.boxplot(data=df[df['price'] < 500], x='room_type', y='price', order=order, ax=ax)
ax.set_title('Price by Room Type (capped at $500 for visibility)')
ax.set_xlabel('Room Type')
ax.set_ylabel('Price (USD)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'price_by_room_type.png')
plt.show()

---
## 2. Feature Exploration

In [ ]:
# ── Categorical counts ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['neighbourhood_group'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Listings by Borough (neighbourhood_group)')
axes[0].set_xlabel('Borough')
axes[0].set_ylabel('Number of Listings')
axes[0].tick_params(axis='x', rotation=30)

df['room_type'].value_counts().plot(kind='bar', ax=axes[1], color='darkorange', edgecolor='white')
axes[1].set_title('Listings by Room Type')
axes[1].set_xlabel('Room Type')
axes[1].set_ylabel('Number of Listings')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'categorical_counts.png')
plt.show()

In [ ]:
# ── Numeric feature distributions ────────────────────────────────────────────
numeric_cols = ['minimum_nights', 'number_of_reviews', 'reviews_per_month',
                'calculated_host_listings_count', 'availability_365']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].hist(df[col], bins=60, color='mediumseagreen', edgecolor='white')
    axes[i].set_title(col)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')

axes[-1].set_visible(False)
plt.suptitle('Numeric Feature Distributions', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'numeric_distributions.png')
plt.show()

### Interpretation — Feature Distributions

- **neighbourhood_group:** Manhattan and Brooklyn dominate (~85% of listings combined). Staten Island is barely represented.
- **room_type:** Entire home/apt and Private room are the dominant types. Shared rooms are rare (<3%).
- **minimum_nights:** Heavily skewed; most listings require 1–3 nights but some have extreme values (>300) suggesting long-term rentals masquerading as short-term.
- **number_of_reviews / reviews_per_month:** Right-skewed. Many listings have 0 reviews (new or inactive).
- **calculated_host_listings_count:** Most hosts have 1 listing; a few professional operators manage dozens.
- **availability_365:** Bimodal — listings cluster near 0 (rarely available) and 365 (always available), suggesting two distinct host behaviours.

---
## 3. Feature–Price Relationships

In [ ]:
# ── Median price by borough ───────────────────────────────────────────────────
borough_price = df.groupby('neighbourhood_group')['price'].median().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
borough_price.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Median Price by Borough')
ax.set_xlabel('Borough')
ax.set_ylabel('Median Price (USD)')
ax.tick_params(axis='x', rotation=30)
for bar in ax.patches:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'${bar.get_height():.0f}', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'median_price_by_borough.png')
plt.show()

In [ ]:
# ── Room type × borough heatmap ───────────────────────────────────────────────
pivot = df.pivot_table(values='price', index='neighbourhood_group',
                       columns='room_type', aggfunc='median')

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd', linewidths=0.5, ax=ax)
ax.set_title('Median Price (USD) — Borough × Room Type')
ax.set_xlabel('Room Type')
ax.set_ylabel('Borough')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'price_heatmap_borough_roomtype.png')
plt.show()

### Interpretation — Borough × Room Type

Manhattan entire home/apt listings command the highest median price by a large margin. The interaction between `room_type` and `neighbourhood_group` is strong — an entire home in Manhattan is priced ~3× higher than an entire home in the Bronx. This interaction should be **explicitly modelled** in Task 03 (e.g., as a multiplication feature or via tree-based models that capture it naturally).

In [ ]:
# ── Geographic scatter — price ────────────────────────────────────────────────
df_plot = df[df['price'] < 500].copy()

fig, ax = plt.subplots(figsize=(10, 8))
sc = ax.scatter(df_plot['longitude'], df_plot['latitude'],
                c=df_plot['price'], cmap='plasma', s=1.5, alpha=0.5)
plt.colorbar(sc, ax=ax, label='Price (USD, capped at $500)')
ax.set_title('Geographic Distribution of Listing Price')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'geo_price_scatter.png')
plt.show()

### Interpretation — Geographic Price Patterns

Price is **spatially structured**: central Manhattan (Midtown, Upper West/East Side) shows consistently high prices. Brooklyn's north-west (Williamsburg, DUMBO) also shows elevated prices. The Bronx and Staten Island are uniformly lower. Raw latitude/longitude carry real signal and should be included as features. Clustering on coordinates (e.g., k-means neighbourhood clusters) could be a useful engineered feature for Task 04.

In [ ]:
# ── Correlation with log1p(price) ─────────────────────────────────────────────
df['log_price'] = np.log1p(df['price'])

corr_cols = ['minimum_nights', 'number_of_reviews', 'reviews_per_month',
             'calculated_host_listings_count', 'availability_365',
             'latitude', 'longitude']

corr = df[corr_cols + ['log_price']].corr()['log_price'].drop('log_price').sort_values()

fig, ax = plt.subplots(figsize=(9, 5))
corr.plot(kind='barh', ax=ax, color=['tomato' if v < 0 else 'steelblue' for v in corr])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Pearson Correlation with log1p(price)')
ax.set_xlabel('Correlation coefficient')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'correlation_with_log_price.png')
plt.show()

print(corr.round(3))

In [ ]:
# ── Top neighbourhoods by median price ────────────────────────────────────────
top_n = df.groupby('neighbourhood')['price'].agg(['median', 'count'])
top_n = top_n[top_n['count'] >= 20].sort_values('median', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(12, 6))
top_n['median'].plot(kind='bar', ax=ax, color='darkorange', edgecolor='white')
ax.set_title('Top 20 Neighbourhoods by Median Price (min 20 listings)')
ax.set_xlabel('Neighbourhood')
ax.set_ylabel('Median Price (USD)')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'top_neighbourhoods_price.png')
plt.show()

In [ ]:
# ── Numeric pairplot (sample for speed) ──────────────────────────────────────
sample = df.sample(n=min(2000, len(df)), random_state=SEED)
pair_cols = ['log_price', 'minimum_nights', 'number_of_reviews',
             'availability_365', 'calculated_host_listings_count']

pg = sns.pairplot(sample[pair_cols], diag_kind='kde', plot_kws={'alpha': 0.3, 's': 10})
pg.fig.suptitle('Pairplot — Numeric Features vs log1p(price)', y=1.01)
pg.savefig(OUTPUT_DIR / 'pairplot_numeric.png')
plt.show()

---
## 4. Remaining Data Quality Decisions

In [ ]:
# ── Identify extreme price outliers ──────────────────────────────────────────
q99 = df['price'].quantile(0.99)
q01 = df['price'].quantile(0.01)
extreme_high = df[df['price'] > q99]

print(f'1st percentile price : ${q01:.0f}')
print(f'99th percentile price: ${q99:.0f}')
print(f'Listings above 99th  : {len(extreme_high)} ({len(extreme_high)/len(df)*100:.2f}%)')
print(f'\nMax price: ${df["price"].max()}')
extreme_high[['neighbourhood_group', 'room_type', 'price']].sort_values('price', ascending=False).head(10)

In [ ]:
# ── Extreme minimum_nights ────────────────────────────────────────────────────
print('minimum_nights > 365:', (df['minimum_nights'] > 365).sum())
print('minimum_nights > 30 :', (df['minimum_nights'] > 30).sum(),
      f'({(df["minimum_nights"] > 30).mean()*100:.1f}% of listings)')
print(df['minimum_nights'].describe().round(1))

In [ ]:
# ── Decision: cap price at 99th percentile; cap minimum_nights at 30 ─────────
PRICE_CAP  = df['price'].quantile(0.99)
NIGHTS_CAP = 30

df_clean = df.copy()
rows_before = len(df_clean)

df_clean = df_clean[df_clean['price'] <= PRICE_CAP]
df_clean['minimum_nights'] = df_clean['minimum_nights'].clip(upper=NIGHTS_CAP)

rows_after = len(df_clean)
print(f'Rows removed (price > ${PRICE_CAP:.0f}): {rows_before - rows_after}')
print(f'Rows remaining: {rows_after}')
print(f'minimum_nights clipped to max {NIGHTS_CAP}')

### Data Quality Decisions

**Price outliers (> 99th percentile, ~$600+):**  
These are genuine luxury listings but represent <1% of data and exert disproportionate influence on regression loss. **Decision: remove rows above the 99th percentile.** This is a modelling choice, not a data error — it allows the model to generalise to the typical price range. These rows should be noted as a limitation.

**minimum_nights > 30:**  
Listings requiring >30 nights are effectively long-term rentals and behave differently from short-term Airbnb listings. Rather than dropping them, **minimum_nights is clipped at 30** to reduce the influence of extreme values while retaining the rows.

**No further missingness** was found beyond what Task 01 handled.

In [ ]:
# ── Drop helper column before saving ─────────────────────────────────────────
df_clean = df_clean.drop(columns=['log_price'])

out_path = OUTPUT_DIR / 'eda_cleaned.csv'
df_clean.to_csv(out_path, index=False)
print(f'Saved eda_cleaned.csv  shape={df_clean.shape}')
df_clean.head()

---
## 5. EDA Summary

### Most Important Features

| Feature | Evidence | Notes |
|---|---|---|
| `room_type` | Large median price gap across categories | Essential; encode as dummy/ordinal |
| `neighbourhood_group` | Clear borough-level price hierarchy | Encode as dummy |
| `neighbourhood` | Fine-grained spatial signal | High cardinality — consider target encoding or grouping |
| `latitude` / `longitude` | Spatial price gradient visible in scatter | Include as numeric; or engineer geo clusters |
| `availability_365` | Bimodal; availability ≈ 0 may flag inactive listings | Retain as-is |
| `calculated_host_listings_count` | Proxy for professional vs casual host | Retain |
| `minimum_nights` | Captures short vs long-term rental signal | Clip at 30 |

### Target Variable Preprocessing
- **Use `log1p(price)` as the regression target** — raw price is too skewed for linear models.
- Back-transform with `np.expm1()` for RMSE evaluation in the original price space.

### Feature Engineering Ideas for Task 04
1. **Room × Borough interaction** — encode as a combined categorical or multiplicative term.
2. **Geo clusters** — k-means on (latitude, longitude) to create a `geo_cluster` feature capturing neighbourhood price zones beyond the listed neighbourhood names.
3. **Review activity flag** — binary indicator: `has_reviews = (number_of_reviews > 0)`.
4. **Professional host flag** — binary: `is_professional_host = (calculated_host_listings_count > 5)`.
5. **Availability bucket** — convert `availability_365` into a 3-category bucket (low/medium/high availability).
6. **log1p transforms** on skewed numerics: `minimum_nights`, `number_of_reviews`, `calculated_host_listings_count`.

In [ ]:
# ── Save EDA summary to outputs/eda_summary.md ───────────────────────────────
summary_text = """# EDA Summary — NYC Airbnb 2019

## Target Variable
- `price` is strongly right-skewed (skewness > 4 raw).
- log1p(price) is approximately normal — **use log1p(price) as the regression target in Task 03**.
- After EDA cleaning, price is capped at the 99th percentile; rows above this threshold are removed.

## Most Predictive Features
1. `room_type` — largest single-feature price differentiator.
2. `neighbourhood_group` (borough) — clear price hierarchy: Manhattan > Brooklyn > Queens > Staten Island > Bronx.
3. `neighbourhood` — fine-grained; high cardinality (200+ values); use target encoding or grouping.
4. `latitude` / `longitude` — spatial price gradient, especially within Manhattan.
5. `availability_365` — bimodal; reflects host commitment level.
6. `calculated_host_listings_count` — proxy for professional hosts who price more strategically.

## Redundant / Problematic Features
- `reviews_per_month` and `number_of_reviews` are correlated; consider dropping one.
- `minimum_nights` was clipped at 30 to remove long-term rental outliers.

## Data Cleaning Applied in Task 02
- Removed listings with price > 99th percentile (extreme luxury outliers).
- Clipped `minimum_nights` at 30 nights.
- Output: `eda_cleaned.csv` (this is the input for Task 03).

## Feature Engineering Recommendations for Task 04
1. Room × Borough interaction term.
2. Geo-cluster feature (k-means on latitude/longitude).
3. Binary `has_reviews` and `is_professional_host` flags.
4. Availability bucket (low/medium/high).
5. log1p transform on skewed numeric features.
"""

summary_path = OUTPUT_DIR / 'eda_summary.md'
summary_path.write_text(summary_text)
print(f'Saved → eda_summary.md')

In [ ]:
# ── Final output checklist ────────────────────────────────────────────────────
expected = [
    'price_distribution.png',
    'price_by_room_type.png',
    'categorical_counts.png',
    'numeric_distributions.png',
    'median_price_by_borough.png',
    'price_heatmap_borough_roomtype.png',
    'geo_price_scatter.png',
    'correlation_with_log_price.png',
    'top_neighbourhoods_price.png',
    'pairplot_numeric.png',
    'eda_cleaned.csv',
    'eda_summary.md',
]

print('Output checklist:')
all_ok = True
for f in expected:
    exists = (OUTPUT_DIR / f).exists()
    status = '  PASS' if exists else '  MISSING'
    print(f'  {status}  {f}')
    if not exists:
        all_ok = False

print()
print('All outputs present!' if all_ok else 'WARNING: some outputs are missing — re-run cells above.')